# Programowanie funkcyjne w Kotlinie i Scali

## Cele

Po przerobieniu notebooka student potrafi tworzyć funkcje czyste i wyższego rzędu, unikać mutacji stanu, używać `map`, `filter`, `fold`, `groupBy`, komponować funkcje, obsługiwać brak wartości i błędy oraz stosować przetwarzanie leniwe.

## Uruchomienie

1. Otwórz plik w JupyterLab / Jupyter Notebook i wybierz **kernel Python 3**.
2. Uruchom **następną komórkę konfiguracyjną**. Rejestruje ona magię `%%kotlin` i `%%scala`.
3. Do uruchamiania przykładów Kotlin potrzebny jest **Kotlin CLI (`kotlinc`, 1.9+)**, a Scali — **Scala CLI (`scala-cli`, Scala 3)**. Same przykłady można czytać bez obu narzędzi.
4. Każda komórka językowa jest **niezależnym skryptem** — można uruchamiać ją osobno. Pierwsze uruchomienie Scali może pobierać zależności z Internetu.

**Uwaga:** jest to jeden notebook porównujący oba języki, dlatego wykorzystuje kernel Python jako prosty uruchamiacz zewnętrznych kompilatorów, a nie mieszane kernele Kotlin/Scala.

In [4]:
# Ten bootstrap używa kernela Python wyłącznie do uruchamiania kompilatorów Kotlin i Scala.
# Komórki %%kotlin oraz %%scala zawierają prawdziwy, samodzielny kod docelowego języka.
import shutil
import subprocess
import tempfile
from pathlib import Path
from IPython.core.magic import register_cell_magic


def _run(source, suffix, command, language):
    if command is None:
        print(f"⚠ {language}: brak kompilatora/uruchamiacza w PATH. Kod jest nadal dostępny do nauki.")
        if language == 'Kotlin':
            print('Zainstaluj Kotlin CLI (polecenie: kotlinc).')
        else:
            print('Zainstaluj Scala CLI (polecenie: scala-cli): https://scala-cli.virtuslab.org/')
        return
    with tempfile.TemporaryDirectory(prefix='fp_course_') as temp_dir:
        src = Path(temp_dir) / f"Example{suffix}"
        src.write_text(source, encoding='utf-8')
        args = command(src)
        try:
            result = subprocess.run(args, cwd=temp_dir, capture_output=True, text=True, timeout=120)
        except subprocess.TimeoutExpired:
            print(f"⚠ {language}: przekroczono limit czasu 120 s (pierwsze uruchomienie może pobierać zależności).")
            return
        if result.stdout.strip():
            print(result.stdout.rstrip())
        if result.stderr.strip():
            print(result.stderr.rstrip())
        if result.returncode != 0:
            print(f"❌ {language}: kod zakończył się błędem (exit code: {result.returncode}).")


@register_cell_magic
def kotlin(line, cell):
    compiler = shutil.which('kotlinc')
    cmd = (lambda p: [compiler, '-script', str(p)]) if compiler else None
    _run(cell, '.kts', cmd, 'Kotlin')


@register_cell_magic
def scala(line, cell):
    cli = shutil.which('scala-cli')
    # Scala CLI uruchamia skrypty .sc bez dodatkowego obiektu @main.
    cmd = (lambda p: [cli, 'run', str(p), '--server=false', '--quiet']) if cli else None
    _run(cell, '.sc', cmd, 'Scala')

print('Środowisko gotowe.')
print('Kotlin CLI:', shutil.which('kotlinc') or 'brak')
print('Scala CLI:', shutil.which('scala-cli') or 'brak')

Środowisko gotowe.
Kotlin CLI: /opt/kotlin/kotlinc/bin/kotlinc
Scala CLI: /usr/local/bin/scala-cli


## Szybkie porównanie konstrukcji

| Operacja | Kotlin | Scala 3 |
|---|---|---|
| Wartość niemutowalna | `val x = 3` | `val x = 3` |
| Funkcja | `fun f(x: Int) = x * 2` | `def f(x: Int) = x * 2` |
| Lambda | `{ x: Int -> x * 2 }` | `(x: Int) => x * 2` |
| Typ funkcji | `(Int) -> Int` | `Int => Int` |
| Transformacja | `map { it * 2 }` | `map(_ * 2)` |
| Selekcja | `filter { it > 0 }` | `filter(_ > 0)` |
| Agregacja | `fold(0) { a, x -> a + x }` | `foldLeft(0)(_ + _)` |
| Brak wartości | `T?` | `Option[T]` |
| Błąd w typie | `Result<T>` | `Either[E, T]` |
| Przetwarzanie leniwe | `Sequence<T>` | `LazyList[T]` |

W Kotlinie `val` chroni tylko powiązanie zmiennej — nie gwarantuje, że obiekt wskazywany przez zmienną jest niemutowalny (np. `MutableList`).

## 1. Czyste funkcje i oddzielenie efektów ubocznych

Czysta funkcja zwraca wynik zależny wyłącznie od argumentów. Wyświetlanie odbywa się poza logiką obliczeń. Przy wielokrotnym wywołaniu dla tych samych parametrów wynik jest taki sam.

**Kotlin**

In [5]:
%%kotlin
fun discountedPrice(price: Double, rate: Double): Double = price * (1.0 - rate)
val result = discountedPrice(200.0, 0.15)
println(result) // 170.0

170.0


**Scala 3**

In [6]:
%%scala
def discountedPrice(price: Double, rate: Double): Double = price * (1.0 - rate)
val result = discountedPrice(200.0, 0.15)
println(result) // 170.0

170.0


**Sprawdź:** zmień zniżkę na `0.25`. Czy funkcja modyfikuje jakikolwiek zewnętrzny stan?

## 2. Niemutowalne dane — `val`, `data class` i `case class`

`val` blokuje ponowne przypisanie do zmiennej. Niemutowalny model danych i metoda `copy` pozwalają wytworzyć nową wersję obiektu bez zmiany starej.

**Kotlin**

In [7]:
%%kotlin
data class Task(val id: Int, val title: String, val done: Boolean)
val initial = Task(1, "Przeczytać rozdział", false)
val completed = initial.copy(done = true)
println(initial)   // done=false
println(completed) // done=true

Task(id=1, title=Przeczytać rozdział, done=false)
Task(id=1, title=Przeczytać rozdział, done=true)


**Scala 3**

In [8]:
%%scala
case class Task(id: Int, title: String, done: Boolean)
val initial = Task(1, "Przeczytać rozdział", false)
val completed = initial.copy(done = true)
println(initial)   // done=false
println(completed) // done=true

Task(1,Przeczytać rozdział,false)
Task(1,Przeczytać rozdział,true)


**Zwróć uwagę:** kopia zmienia jedno pole, ale oryginał pozostaje nietknięty.

## 3. Funkcje wyższego rzędu

Funkcja wyższego rzędu przyjmuje inne funkcje jako argumenty albo je zwraca. Zamiast duplikować algorytm, przekazujemy tylko zmienny fragment operacji.

**Kotlin**

In [9]:
%%kotlin
fun applyToAll(numbers: List<Int>, transform: (Int) -> Int): List<Int> =
    numbers.map(transform)
val doubled = applyToAll(listOf(1, 2, 3)) { it * 2 }
val squared = applyToAll(listOf(1, 2, 3)) { it * it }
println(doubled) // [2, 4, 6]
println(squared) // [1, 4, 9]

[2, 4, 6]
[1, 4, 9]


**Scala 3**

In [10]:
%%scala
def applyToAll(numbers: List[Int], transform: Int => Int): List[Int] =
  numbers.map(transform)
val doubled = applyToAll(List(1, 2, 3), x => x * 2)
val squared = applyToAll(List(1, 2, 3), x => x * x)
println(doubled) // List(2, 4, 6)
println(squared) // List(1, 4, 9)

List(2, 4, 6)
List(1, 4, 9)


**Ćwiczenie w trakcie:** przekaż funkcję zwiększającą każdy element o 10.

## 4. Domknięcia i lambdy

Lambda może korzystać ze zmiennych z otaczającego ją zakresu. Takie powiązanie funkcji z kontekstem to **domknięcie** (closure).

**Kotlin**

In [11]:
%%kotlin
val minimum = 10
val isGreaterThanMinimum: (Int) -> Boolean = { number -> number > minimum }
println(listOf(5, 10, 11, 30).filter(isGreaterThanMinimum)) // [11, 30]

[11, 30]


**Scala 3**

In [12]:
%%scala
val minimum = 10
val isGreaterThanMinimum: Int => Boolean = number => number > minimum
println(List(5, 10, 11, 30).filter(isGreaterThanMinimum)) // List(11, 30)

List(11, 30)


**Ważne:** domknięcia mogą przechwytywać stan mutowalny, lecz w kodzie funkcyjnym zwykle preferujemy niezmienność.

## 5. Potoki `filter` → `map` → `sorted`

Operacje kolekcji tworzą czytelny potok transformacji. `filter` wybiera dane, `map` je przekształca, a `sorted` porządkuje wynik.

**Kotlin**

In [13]:
%%kotlin
val values = listOf(7, -3, 4, 2, -8, 1)
val result = values.filter { it > 0 }.map { it * it }.sorted()
println(result) // [1, 4, 16, 49]

[1, 4, 16, 49]


**Scala 3**

In [14]:
%%scala
val values = List(7, -3, 4, 2, -8, 1)
val result = values.filter(_ > 0).map(x => x * x).sorted
println(result) // List(1, 4, 16, 49)

List(1, 4, 16, 49)


**Pytanie:** w którym miejscu znajdują się elementy liczb ujemnych?

## 6. `fold` i `reduce` — agregacja elementów

`fold` posiada wartość początkową i działa także na pustej kolekcji; `reduce` potrzebuje niepustej kolekcji. Zapis funkcji agregującej jest jawny.

**Kotlin**

In [15]:
%%kotlin
val numbers = listOf(2, 3, 4)
println(numbers.fold(0) { acc, n -> acc + n }) // 9
println(numbers.fold(1) { acc, n -> acc * n }) // 24
println(emptyList<Int>().fold(0) { acc, n -> acc + n }) // 0
println(numbers.reduce { acc, n -> acc + n }) // 9

9
24
0
9


**Scala 3**

In [16]:
%%scala
val numbers = List(2, 3, 4)
println(numbers.foldLeft(0)((acc, n) => acc + n)) // 9
println(numbers.foldLeft(1)((acc, n) => acc * n)) // 24
println(List.empty[Int].foldLeft(0)((acc, n) => acc + n)) // 0
println(numbers.reduce((acc, n) => acc + n)) // 9

9
24
0
9


**Pytanie:** dlaczego w iloczynie zaczynamy od `1`, a w sumie od `0`?

## 7. Grupowanie i agregowanie danych

`groupBy` tworzy grupy według klucza. Z każdej grupy można obliczyć sumę, liczność lub inne statystyki bez ręcznych pętli.

**Kotlin**

In [17]:
%%kotlin
data class Purchase(val category: String, val amount: Int)
val purchases = listOf(Purchase("book", 30), Purchase("food", 15), Purchase("book", 20))
val totals = purchases.groupBy { it.category }
    .mapValues { (_, group) -> group.sumOf { it.amount } }
println(totals.toSortedMap()) // {book=50, food=15}

{book=50, food=15}


**Scala 3**

In [18]:
%%scala
case class Purchase(category: String, amount: Int)
val purchases = List(Purchase("book", 30), Purchase("food", 15), Purchase("book", 20))
val totals = purchases.groupBy(_.category)
  .view.mapValues(group => group.map(_.amount).sum).toMap
println(totals.toSeq.sortBy(_._1)) // List((book,50), (food,15))

List((book,50), (food,15))


**Wskazówka:** w Kotlinie `mapValues`, a w Scali `view.mapValues(...).toMap` tworzą nowe mapy wynikowe.

## 8. Kompozycja funkcji

Funkcje można łączyć bez zmiennych tymczasowych. Kompozycja `compose(f, g)` oznacza zastosowanie `g`, a potem `f`.

**Kotlin**

In [19]:
%%kotlin
fun <A, B, C> compose(f: (B) -> C, g: (A) -> B): (A) -> C =
    { value -> f(g(value)) }
val plusThree: (Int) -> Int = { it + 3 }
val double: (Int) -> Int = { it * 2 }
val transform = compose(double, plusThree)
println(transform(4)) // 14

14


**Scala 3**

In [20]:
%%scala
def compose[A, B, C](f: B => C, g: A => B): A => C =
  value => f(g(value))
val plusThree: Int => Int = _ + 3
val double: Int => Int = _ * 2
val transform = compose(double, plusThree)
println(transform(4)) // 14

14


**Porównaj:** zamień argumenty w `compose` — wynikiem będzie `11`, a nie `14`.

## 9. Currying i częściowa aplikacja

Funkcja może zwracać kolejną funkcję, co pozwala związać część parametrów z góry i zbudować wyspecjalizowaną operację.

**Kotlin**

In [21]:
%%kotlin
fun add(a: Int): (Int) -> Int = { b -> a + b }
val plusTen = add(10)
println(plusTen(2)) // 12
println(listOf(1, 2, 3).map(plusTen)) // [11, 12, 13]

12
[11, 12, 13]


**Scala 3**

In [22]:
%%scala
def add(a: Int): Int => Int = b => a + b
val plusTen = add(10)
println(plusTen(2)) // 12
println(List(1, 2, 3).map(plusTen)) // List(11, 12, 13)

12
List(11, 12, 13)


**Zastosowanie:** przygotowanie funkcji z ustaloną stawką VAT lub współczynnikiem skalowania.

## 10. Brak wartości: nullable vs `Option`

Kotlin przedstawia brak wartości jako typ nullable (`User?`) oraz operatory `?.` i `?:`. Scala standardowo stosuje typ `Option`, którego wariantami są `Some` oraz `None`.

**Kotlin**

In [23]:
%%kotlin
data class User(val id: Int, val name: String)
val users = listOf(User(1, "Ala"), User(2, "Olek"))
fun findName(id: Int): String? = users.find { it.id == id }?.name
println(findName(2)?.uppercase() ?: "nie znaleziono") // OLEK
println(findName(9)?.uppercase() ?: "nie znaleziono") // nie znaleziono

OLEK
nie znaleziono


**Scala 3**

In [24]:
%%scala
case class User(id: Int, name: String)
val users = List(User(1, "Ala"), User(2, "Olek"))
def findName(id: Int): Option[String] = users.find(_.id == id).map(_.name)
println(findName(2).map(_.toUpperCase).getOrElse("nie znaleziono")) // OLEK
println(findName(9).map(_.toUpperCase).getOrElse("nie znaleziono")) // nie znaleziono

OLEK
nie znaleziono


**Ważne:** jawne modelowanie braku danych chroni przed częścią błędów w czasie wykonania.

## 11. Błędy jako wartości — `Result` i `Either`

Zamiast niespodziewanie rzucać wyjątek w dalszej części potoku, można zawrzeć powodzenie lub błąd w typie wartości i transformować poprawne wyniki.

**Kotlin**

In [25]:
%%kotlin
fun parsePositive(text: String): Result<Int> = runCatching {
    val number = text.toInt()
    require(number > 0) { "Liczba musi być dodatnia" }
    number
}
println(parsePositive("12").map { it * 2 }.getOrElse { -1 }) // 24
println(parsePositive("-3").map { it * 2 }.getOrElse { -1 }) // -1

24
-1


**Scala 3**

In [26]:
%%scala
def parsePositive(text: String): Either[String, Int] =
  text.toIntOption.toRight("To nie jest liczba")
    .flatMap(n => if n > 0 then Right(n) else Left("Liczba musi być dodatnia"))
println(parsePositive("12").map(_ * 2)) // Right(24)
println(parsePositive("-3").map(_ * 2)) // Left(Liczba musi być dodatnia)

Right(24)
Left(Liczba musi być dodatnia)


**Porównaj:** Scala używa tu `Either[String, Int]`, a Kotlin `Result<Int>`.

## 12. Dopasowanie wzorców i typy sumacyjne

Typy `sealed` pozwalają opisać skończony zestaw przypadków. Kompilator pomaga sprawdzić, czy obsłużono je wszystkie.

**Kotlin**

In [27]:
%%kotlin
sealed interface Payment {
    data class Accepted(val reference: String) : Payment
    data class Rejected(val reason: String) : Payment
}
fun display(payment: Payment): String = when (payment) {
    is Payment.Accepted -> "OK: ${payment.reference}"
    is Payment.Rejected -> "Błąd: ${payment.reason}"
}
println(display(Payment.Accepted("TX-42"))) // OK: TX-42

OK: TX-42


**Scala 3**

In [28]:
%%scala
enum Payment:
  case Accepted(reference: String)
  case Rejected(reason: String)

def display(payment: Payment): String = payment match
  case Payment.Accepted(reference) => s"OK: $reference"
  case Payment.Rejected(reason) => s"Błąd: $reason"
println(display(Payment.Accepted("TX-42"))) // OK: TX-42

OK: TX-42


**Zastosowanie:** wyniki transakcji, walidacja danych, stany operacji asynchronicznych.

## 13. Obliczenia leniwe — `Sequence` i `LazyList`

Kolekcja leniwa oblicza elementy dopiero wtedy, gdy są potrzebne. Można dzięki temu bezpiecznie opisać potencjalnie nieskończony strumień i pobrać jego skończony fragment.

**Kotlin**

In [29]:
%%kotlin
val squares = generateSequence(1) { it + 1 }
    .filter { it % 2 == 0 }
    .map { it * it }
    .take(5)
    .toList()
println(squares) // [4, 16, 36, 64, 100]

[4, 16, 36, 64, 100]


**Scala 3**

In [30]:
%%scala
val squares = LazyList.from(1)
  .filter(_ % 2 == 0)
  .map(x => x * x)
  .take(5)
  .toList
println(squares) // List(4, 16, 36, 64, 100)

List(4, 16, 36, 64, 100)


**Uwaga:** nie próbuj obliczać `toList()` na nieskończonym strumieniu bez `take` lub innego warunku końca.

## 14. Rekurencja ogonowa i przykład końcowy

Rekurencja ogonowa (`tailrec` / `@tailrec`) pozwala kompilatorowi zoptymalizować serię wywołań. Na końcu przykład przygotowania wyniku z funkcji agregujących.

**Kotlin**

In [31]:
%%kotlin
tailrec fun factorial(n: Int, acc: Long = 1L): Long {
    require(n >= 0)
    return if (n <= 1) acc else factorial(n - 1, acc * n)
}
println(factorial(5)) // 120

data class Sale(val category: String, val amount: Int, val paid: Boolean)
val sales = listOf(Sale("book", 20, true), Sale("food", 15, false), Sale("book", 30, true))
val summary = sales.filter { it.paid }.groupBy { it.category }
    .mapValues { (_, group) -> group.sumOf { it.amount } }
println(summary.toSortedMap()) // {book=50}

120
{book=50}


**Scala 3**

In [32]:
%%scala
@annotation.tailrec
def factorial(n: Int, acc: Long = 1L): Long =
  require(n >= 0)
  if n <= 1 then acc else factorial(n - 1, acc * n)
println(factorial(5)) // 120

case class Sale(category: String, amount: Int, paid: Boolean)
val sales = List(Sale("book", 20, true), Sale("food", 15, false), Sale("book", 30, true))
val summary = sales.filter(_.paid).groupBy(_.category)
  .view.mapValues(group => group.map(_.amount).sum).toMap
println(summary.toSeq.sortBy(_._1)) // List((book,50))

120
List((book,50))


**Ograniczenie:** `factorial` na typie `Long` przepełnia się przy większych `n` (np. `21`).